In [ ]:
# Import libraries
from pyspark.sql import SparkSession
from lib.helpers import get_run_date
from lib.job_control import get_max_insert_ts, insert_log
from pyspark.sql.functions import current_timestamp, lit, first
from lib.aws_s3 import warehouse_dir
import requests

In [ ]:
# Create the SparkSession
"""
spark = (
    SparkSession.builder
    .appName("18 - Vehicle Production")
    .config("spark.sql.warehouse.dir", warehouse_dir)
    .enableHiveSupport()
    .getOrCreate()
)

spark
"""

In [ ]:
# Job parameters
now_ts = spark.range(1).select(current_timestamp().alias("now")).first()["now"]
run_date = get_run_date()
table = "vehicle"
schema_name = "prd"
table_name = f"dim_{table}"
table_full_name = f"{schema_name}.{table_name}"

print("****************************************")
print("18 - Vehicle Production")
print("****************************************")
print(f"now_ts = {now_ts}")
print(f"run_date = {run_date}")
print(f"table = {table}")
print(f"schema_name = {schema_name}")
print(f"table_name = {table_name}")
print(f"table_full_name = {table_full_name}")
print("")

In [ ]:
# Get max insert_ts for {schema_name}.{table_name} table from prd.job_control
max_insert_ts = get_max_insert_ts(spark, schema_name, table_name)
print(f"Max insert_ts in {table_full_name}: {max_insert_ts}\n")

In [ ]:
# Read VINs from prd.dim_vehicle
df_vehicle = (
    spark
    .table("prd.dim_vehicle")
).select('vin').distinct()

In [ ]:
# Read VINs from prd.fact_service
df_service = (
    spark
    .table("prd.fact_service")
).select('vin').distinct()

In [ ]:
# Get new VINs from prd.fact_service but not in prd.dim_vehicle
df_new_vins = df_service.join(df_vehicle, on="VIN", how="left_anti")
print(f"{df_new_vins.count()} new VINs identified!\n")

In [ ]:
# Procced only if there are VINs to decode

if df_new_vins.count() > 0:
    print("Fetching data from VIN decoder!\n")
    
    # Store the new VINs in an array
    vins = [row["VIN"] for row in df_new_vins.select("VIN").collect()]
    
    # Get VINs info from an online VIN decoder
    url_template = "https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVin/{}?format=json"
    schema = "VIN string, Variable string, Value string"
    rows = []
    
    with requests.Session() as session:
        for vin in vins:
            try:
                response = session.get(
                    url_template.format(vin),
                    timeout=30
                )
                response.raise_for_status()
    
                data = response.json()
    
                for item in data.get("Results", []):
                    rows.append((
                        vin,
                        item.get("Variable"),
                        item.get("Value")
                    ))
    
            except (requests.RequestException, ValueError) as e:
                print(f"Failed to process VIN {vin}: {e}")
    
    # Create the PySpark DataFrame from the returned data
    df_raw = spark.createDataFrame(rows, schema=schema)
    
    # Tranpose so the output matches the schema of prd.dim_vehicle
    df_pivot = (
        df_raw.groupBy("VIN")
          .pivot("Variable", [
              "Make",
              "Model",
              "Model Year",
              "Trim",
              "Fuel Type - Primary",
              "Drive Type"
          ])
          .agg(first("Value"))
          .withColumnRenamed("Make", "make")
          .withColumnRenamed("Model", "model")
          .withColumnRenamed("Model Year", "model_year")
          .withColumnRenamed("Trim", "trim")
          .withColumnRenamed("Fuel Type - Primary", "fuel_type")
          .withColumnRenamed("Drive Type", "drive_type")
          .select("VIN", "make", "model", "model_year", "trim", "fuel_type", "drive_type")
    )


    print(f"df_pivot: {df_pivot.count()} records\n")
    df_pivot.show()
    df_pivot.printSchema()

    # Add audit columns insert_dt and run_date
    df_prd = (
        df_pivot
        .withColumn("insert_ts", lit(now_ts))
    )

    print("Added audit columns")
    print(f"df_prd: {df_prd.count()} records")
    df_prd.show()
    df_prd.printSchema()

    # Load data to prd.dim_vehicle
    (
        df_prd
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(table_full_name)
    )

    print(f"Data loaded in this run:")
    spark.table(f'{schema_name}.dim_{table}').where(f"insert_ts = '{now_ts}'").show()
    
    # Add entry to prd.job_control
    if insert_log(spark, schema_name, table_name, now_ts, run_date):
        print(f"{schema_name}.{table_name} entry added to prd.job_control")
    
    # Review added entry
    spark.sql(f"select * from prd.job_control where table_name = '{table_name}' order by insert_ts desc limit 1").show(truncate=False)
else:
    print("No new VINs to decode today!")

In [ ]:
# Stop spark session
#spark.stop()